<a href="https://colab.research.google.com/github/SaimNaveed646/PredictFutureSales/blob/main/PredictFutureSales.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
df = pd.read_csv('/content/drive/MyDrive/MLProject/FutureSale/sales_train.csv')

In [ ]:
df.shape

(2935849, 6)

In [ ]:
df.columns

Index(['date', 'date_block_num', 'shop_id', 'item_id', 'item_price',
       'item_cnt_day'],
      dtype='object')

In [ ]:
df.head()

,date,date_block_num,shop_id,item_id,item_price,item_cnt_day
0,02.01.2013,0,59,22154,999.00,1.0
1,03.01.2013,0,25,2552,899.00,1.0
2,05.01.2013,0,25,2552,899.00,-1.0
3,06.01.2013,0,25,2554,1709.05,1.0
4,15.01.2013,0,25,2555,1099.00,1.0


In [ ]:
df['shop_id'].nunique()

60

In [ ]:
df['item_id'].nunique()

21807

In [ ]:
df['date_block_num'].nunique()

34

In [ ]:
df.isnull().sum()

,0
date,0
date_block_num,0
shop_id,0
item_id,0
item_price,0
item_cnt_day,0


In [ ]:
df.describe()

,date_block_num,shop_id,item_id,item_price,item_cnt_day
count,2.935849e+06,2.935849e+06,2.935849e+06,2.935849e+06,2.935849e+06
mean,1.456991e+01,3.300173e+01,1.019723e+04,8.908532e+02,1.242641e+00
std,9.422988e+00,1.622697e+01,6.324297e+03,1.729800e+03,2.618834e+00
min,0.000000e+00,0.000000e+00,0.000000e+00,-1.000000e+00,-2.200000e+01
25%,7.000000e+00,2.200000e+01,4.476000e+03,2.490000e+02,1.000000e+00
50%,1.400000e+01,3.100000e+01,9.343000e+03,3.990000e+02,1.000000e+00
75%,2.300000e+01,4.700000e+01,1.568400e+04,9.990000e+02,1.000000e+00
max,3.300000e+01,5.900000e+01,2.216900e+04,3.079800e+05,2.169000e+03


-->Negative values in item_cnt_day or item_price represent product returns or refunds, where previously sold items were returned by customers, resulting in negative sales entries in the dataset.We are
removing the returns or refunds features.

In [ ]:
df = df[df['item_cnt_day'] > 0]
df = df[df['item_price'] > 0]

In [ ]:
df.shape

(2928492, 6)

In [ ]:
df.describe()

,date_block_num,shop_id,item_id,item_price,item_cnt_day
count,2.928492e+06,2.928492e+06,2.928492e+06,2.928492e+06,2.928492e+06
mean,1.456976e+01,3.300295e+01,1.020028e+04,8.894668e+02,1.248337e+00
std,9.422951e+00,1.622543e+01,6.324396e+03,1.727499e+03,2.619586e+00
min,0.000000e+00,0.000000e+00,0.000000e+00,7.000000e-02,1.000000e+00
25%,7.000000e+00,2.200000e+01,4.477000e+03,2.490000e+02,1.000000e+00
50%,1.400000e+01,3.100000e+01,9.355000e+03,3.990000e+02,1.000000e+00
75%,2.300000e+01,4.700000e+01,1.569100e+04,9.990000e+02,1.000000e+00
max,3.300000e+01,5.900000e+01,2.216900e+04,3.079800e+05,2.169000e+03


# **Daily → Monthly Conversion**

In [ ]:
monthly = df.groupby(
    ['date_block_num', 'shop_id', 'item_id'],
    as_index=False
)['item_cnt_day'].sum()

In [ ]:
monthly

,date_block_num,shop_id,item_id,item_cnt_day
0,0,0,32,6.0
1,0,0,33,3.0
2,0,0,35,1.0
3,0,0,43,1.0
4,0,0,51,2.0
...,...,...,...,...
1608221,33,59,22087,6.0
1608222,33,59,22088,2.0
1608223,33,59,22091,1.0
1608224,33,59,22100,1.0


In [ ]:
monthly.rename(
    columns={'item_cnt_day': 'item_cnt_month'},
    inplace=True
)

In [ ]:
monthly

,date_block_num,shop_id,item_id,item_cnt_month
0,0,0,32,6.0
1,0,0,33,3.0
2,0,0,35,1.0
3,0,0,43,1.0
4,0,0,51,2.0
...,...,...,...,...
1608221,33,59,22087,6.0
1608222,33,59,22088,2.0
1608223,33,59,22091,1.0
1608224,33,59,22100,1.0


In [ ]:
monthly.shape

(1608226, 4)

After aggregation, monthly rows ≈ 1.6M

**Problem:** Table only has rows where sales occurred.
If an item wasn’t sold in a shop that month → row is missing, but sales were actually 0.
Model must see these zeros too, otherwise it thinks every item sells every month — a common beginner mistake

In [ ]:
for month in monthly['date_block_num'].unique():
  print(month)

0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33


In [ ]:
monthly['shop_id'].unique()

array([ 0,  1,  2,  3,  4,  6,  7,  8, 10, 12, 13, 14, 15, 16, 18, 19, 21,
       22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 35, 37, 38, 41, 42, 43,
       44, 45, 46, 47, 50, 51, 52, 53, 54, 56, 59,  5, 57, 58, 55, 17,  9,
       49, 39, 40, 48, 34, 33, 20, 11, 36])

In [ ]:
monthly[monthly['date_block_num'] == 0]['shop_id'].unique()

array([ 0,  1,  2,  3,  4,  6,  7,  8, 10, 12, 13, 14, 15, 16, 18, 19, 21,
       22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 35, 37, 38, 41, 42, 43,
       44, 45, 46, 47, 50, 51, 52, 53, 54, 56, 59])

In [ ]:
from itertools import product

matrix = []

for month in monthly['date_block_num'].unique():
    shops = monthly[monthly['date_block_num'] == month]['shop_id'].unique()
    items = monthly[monthly['date_block_num'] == month]['item_id'].unique()

    matrix.append(                                                                                         #product([0], [1,2], [10,11]) => [0,1,10],[0,1,11],[0,2,10],[0,2,11]
        np.array(list(product([month], shops, items)), dtype='int16')
    )

matrix = pd.DataFrame(
    np.vstack(matrix),
    columns=['date_block_num', 'shop_id', 'item_id']
)

In [ ]:
matrix = matrix.merge(
    monthly,
    on=['date_block_num', 'shop_id', 'item_id'],
    how='left'
)

**Why merge after Cartesian?**

Cartesian product → creates all possible month–shop–item combinations (structure), but no sales values.

Merge → attaches actual sales from the monthly table. Missing rows → NaN → fill with 0.

Result → every possible combination has a sales value, so model can learn zeros too.

Core idea: Cartesian = structure, Merge = real sales.

In [ ]:
print(matrix['item_cnt_month'].isna())

0           False
1           False
2           False
3           False
4           False
            ...  
10884549    False
10884550    False
10884551    False
10884552    False
10884553    False
Name: item_cnt_month, Length: 10884554, dtype: bool


In [ ]:
matrix

,date_block_num,shop_id,item_id,item_cnt_month
0,0,0,32,6.0
1,0,0,33,3.0
2,0,0,35,1.0
3,0,0,43,1.0
4,0,0,51,2.0
...,...,...,...,...
10884549,33,59,5662,1.0
10884550,33,59,10068,1.0
10884551,33,59,12839,1.0
10884552,33,59,18275,1.0


In [ ]:
matrix['item_cnt_month'] = matrix['item_cnt_month'].fillna(0)

In [ ]:
print("Matrix Shape:", matrix.shape)
matrix.head()

Matrix Shape: (10884554, 4)


,date_block_num,shop_id,item_id,item_cnt_month
0,0,0,32,6.0
1,0,0,33,3.0
2,0,0,35,1.0
3,0,0,43,1.0
4,0,0,51,2.0


# **Lag Features**

**Why we sort before lag features:**

Lag uses shift() → picks previous row’s value.

If data isn’t sorted by shop → item → month, shift will pick wrong month.

Sort ensures for each shop-item, months are in order → lag feature is correct.

Rule: Always sort by group keys + time before creating lag features.

In [ ]:
matrix = matrix.sort_values(
    ['shop_id', 'item_id', 'date_block_num']
)
matrix

,date_block_num,shop_id,item_id,item_cnt_month
372824,1,0,12,0.0
6274,0,0,19,0.0
2385,0,0,27,0.0
370788,1,0,27,0.0
5153,0,0,28,0.0
...,...,...,...,...
360027,0,59,22168,0.0
737917,1,59,22168,0.0
1121756,2,59,22168,0.0
3350802,8,59,22168,0.0


**How lag feature works:**

->Group by shop_id, item_id → each shop-item becomes a mini time-series.

->Select item_cnt_month → only sales values matter.

->Shift(1) → previous month’s value moves to current row.

->Assign → matrix['lag_1'] = ...

First month → NaN (no previous month).

Always sort by shop_id, item_id, date_block_num before shift.

Lag helps model see past month’s sales to predict next month.

In [ ]:
matrix['lag_1'] = matrix.groupby(
    ['shop_id', 'item_id']
)['item_cnt_month'].shift(1)


In [ ]:
matrix['lag_2'] = matrix.groupby(
    ['shop_id', 'item_id']
)['item_cnt_month'].shift(2)


In [ ]:
lags = [3, 6, 12]
for lag in lags:
    matrix[f'lag_{lag}'] = matrix.groupby(['shop_id','item_id'])['item_cnt_month'].shift(lag)

In [ ]:
matrix.fillna(0, inplace=True)

In [ ]:
matrix

,date_block_num,shop_id,item_id,item_cnt_month,lag_1,lag_2,lag_3,lag_6,lag_12
372824,1,0,12,0.0,0.0,0.0,0.0,0.0,0.0
6274,0,0,19,0.0,0.0,0.0,0.0,0.0,0.0
2385,0,0,27,0.0,0.0,0.0,0.0,0.0,0.0
370788,1,0,27,0.0,0.0,0.0,0.0,0.0,0.0
5153,0,0,28,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...
360027,0,59,22168,0.0,0.0,0.0,0.0,0.0,0.0
737917,1,59,22168,0.0,0.0,0.0,0.0,0.0,0.0
1121756,2,59,22168,0.0,0.0,0.0,0.0,0.0,0.0
3350802,8,59,22168,0.0,0.0,0.0,0.0,0.0,0.0


In [ ]:
matrix.shape

(10884554, 9)

In [ ]:
# Monthly avg price per item
monthly_price = df.groupby(['date_block_num','shop_id','item_id'], as_index=False)['item_price'].mean()
monthly_price.rename(columns={'item_price':'avg_item_price'}, inplace=True)

# Merge with main matrix
matrix = matrix.merge(monthly_price, on=['date_block_num','shop_id','item_id'], how='left')
matrix['avg_item_price'].fillna(0, inplace=True)

/tmp/ipython-input-210/699136728.py:7: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  matrix['avg_item_price'].fillna(0, inplace=True)


In [ ]:
matrix

,date_block_num,shop_id,item_id,item_cnt_month,lag_1,lag_2,lag_3,lag_6,lag_12,avg_item_price
0,1,0,12,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0,0,19,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0,0,27,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,1,0,27,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0,0,28,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...
10884549,0,59,22168,0.0,0.0,0.0,0.0,0.0,0.0,0.0
10884550,1,59,22168,0.0,0.0,0.0,0.0,0.0,0.0,0.0
10884551,2,59,22168,0.0,0.0,0.0,0.0,0.0,0.0,0.0
10884552,8,59,22168,0.0,0.0,0.0,0.0,0.0,0.0,0.0


# **Seasonality**
Month — date_block_num % 12 → seasonality indicator

In [ ]:
matrix['month'] = matrix['date_block_num'] % 12        # har month ko represent kar rha ha by modulus

# **Train/Test Split**

In [ ]:
X_train = matrix[matrix['date_block_num'] < 33].drop('item_cnt_month', axis=1)
y_train = matrix[matrix['date_block_num'] < 33]['item_cnt_month']

X_valid = matrix[matrix['date_block_num'] == 33].drop('item_cnt_month', axis=1)
y_valid = matrix[matrix['date_block_num'] == 33]['item_cnt_month']

# **Model Training**

In [ ]:
import lightgbm as lgb

train_data = lgb.Dataset(X_train, label=y_train)
valid_data = lgb.Dataset(X_valid, label=y_valid)



params = {
    'objective':'regression',
    'metric':'rmse',
    'learning_rate':0.1,
    'num_leaves':31,
    'verbose':-1
}

model = lgb.train(
    params,
    train_data,
    valid_sets=[train_data, valid_data],
    num_boost_round=1000,
    callbacks=[
        lgb.early_stopping(stopping_rounds=50)
    ]
)



Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[24]	training's rmse: 2.07012	valid_1's rmse: 4.69555


The model was trained using past sales data, including lag features such as lag_1, lag_2, lag_3, lag_6, and lag_12. Based on these historical patterns, it predicted sales for month 33. The output shape of 238,084 indicates that predictions were generated for every shop_id and item_id combination in that month. If previous lag values were zero, the model typically predicted a value close to zero for that combination.

In [ ]:
y_pred = model.predict(X_valid)
y_pred.shape

(238084,)

In [ ]:
y_pred = np.clip(y_pred, 0, 20)
y_pred.round()

array([0., 2., 0., ..., 0., 0., 0.])

In real life, sales can definitely be greater than 20. However, in the Kaggle Predict Future Sales competition, predictions were evaluated after being clipped to the range of 0 to 20. This means:

If the model predicted 35, it would be treated as 20.

If the model predicted -3, it would be treated as 0.

In [ ]:
pred_df = X_valid[['shop_id','item_id']].copy()
pred_df['predicted_sales'] = y_pred.round()

In [ ]:
pred_df

,shop_id,item_id,predicted_sales
32654,2,30,0.0
32687,2,31,2.0
32721,2,32,0.0
32755,2,33,0.0
32869,2,40,0.0
...,...,...,...
10884483,59,22162,0.0
10884486,59,22163,0.0
10884495,59,22164,0.0
10884522,59,22166,0.0


In [ ]:
shops = pd.read_csv('/content/drive/MyDrive/MLProject/FutureSale/shops.csv', encoding='utf-8')
shops.head()

,shop_name,shop_id
0,"!Якутск Орджоникидзе, 56 фран",0
1,"!Якутск ТЦ ""Центральный"" фран",1
2,"Адыгея ТЦ ""Мега""",2
3,"Балашиха ТРК ""Октябрь-Киномир""",3
4,"Волжский ТЦ ""Волга Молл""",4


In [ ]:
items = pd.read_csv('/content/drive/MyDrive/MLProject/FutureSale/items.csv', encoding='utf-8')
items.head()

,item_name,item_id,item_category_id
0,! ВО ВЛАСТИ НАВАЖДЕНИЯ (ПЛАСТ.) D,0,40
1,!ABBYY FineReader 12 Professional Edition Full...,1,76
2,***В ЛУЧАХ СЛАВЫ (UNV) D,2,40
3,***ГОЛУБАЯ ВОЛНА (Univ) D,3,40
4,***КОРОБКА (СТЕКЛО) D,4,40


In [ ]:
# Strip extra spaces
shops['shop_name'] = shops['shop_name'].str.strip()

# Remove exclamation marks
shops['shop_name'] = shops['shop_name'].str.replace('!', '', regex=False)

In [ ]:
items['item_name'] = items['item_name'].str.strip()
items['item_name'] = items['item_name'].str.replace('!', '', regex=False)

In [ ]:
pred_df = pred_df.drop(columns=['shop_name','item_name'], errors='ignore')


In [ ]:
pred_df = pred_df.merge(shops, on='shop_id', how='left')
pred_df = pred_df.merge(items, on='item_id', how='left')

pred_df[['shop_name','item_name','predicted_sales']].head()

,shop_name,item_name,predicted_sales
0,"Адыгея ТЦ ""Мега""",007: КООРДИНАТЫ «СКАЙФОЛЛ»,0.0
1,"Адыгея ТЦ ""Мега""",007: КООРДИНАТЫ «СКАЙФОЛЛ» (BD),2.0
2,"Адыгея ТЦ ""Мега""",1+1,0.0
3,"Адыгея ТЦ ""Мега""",1+1 (BD),0.0
4,"Адыгея ТЦ ""Мега""",100 Best classical melodies (mp3-CD) (Digipack),0.0


https://chatgpt.com/share/69a034d9-bc7c-8001-a1f6-6f58a50b28f0

https://chatgpt.com/share/69a03575-6eb8-8001-874c-9fde4a34ee0e